# Chapter 3 · Simon's Algorithm

## Objectives

1. Understand Simon's problem and its hidden periodicity structure.
2. Implement Simon's oracle for $n$-bit strings.
3. Verify that the quantum procedure + classical post-processing (linear algebra mod 2) recovers the period $\mathbf{s}$.

---

## 3C.1 The Problem

Given $f: \{0,1\}^n \to \{0,1\}^n$ with the promise that there exists a hidden vector $\mathbf{s} \neq \mathbf{0}$ such that:

$$f(\mathbf{x}) = f(\mathbf{y}) \iff \mathbf{y} = \mathbf{x} \oplus \mathbf{s}$$

Simon's algorithm finds $\mathbf{s}$ using $O(n)$ oracle calls, compared to the classical solution which requires $O(2^{n/2})$ queries.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


def simon_oracle(secret: str) -> QuantumCircuit:
    """Simon's oracle for the hidden period s.

    Implements a periodic function based on the secret string s:
    copies x -> y (cross CNOTs) and XORs with s if the first half matches.

    Parameters
    ----------
    secret : str
        Binary string of length n (the hidden period).
    """
    n = len(secret)
    qc = QuantumCircuit(2 * n, name=f'Simon({secret})')

    # Copy input qubits to the output register
    for i in range(n):
        qc.cx(i, n + i)

    # Apply XOR with s on the first qubit that is '1'
    pivot = next((i for i, b in enumerate(secret) if b == '1'), None)
    if pivot is not None:
        for i, bit in enumerate(secret):
            if bit == '1':
                qc.cx(pivot, n + i)
    return qc


def simon_circuit(secret: str) -> QuantumCircuit:
    """Quantum circuit for Simon's algorithm."""
    n = len(secret)
    qc = QuantumCircuit(2 * n, n)

    # Hadamard on input qubits
    qc.h(range(n))
    qc.barrier()

    # Oracle
    oracle = simon_oracle(secret)
    qc.compose(oracle, inplace=True)
    qc.barrier()

    # Second Hadamard on input qubits
    qc.h(range(n))
    qc.barrier()

    # Measure only the input qubits
    qc.measure(range(n), range(n))
    return qc


def simon_postprocessing(equations: list, n: int) -> str:
    """Solves the linear system mod 2 to find the period s.

    Steps:
    1. Form a matrix A whose rows are the measured vectors y.
    2. Apply Gaussian elimination mod 2 to find the kernel.
    """
    # Convert equations to binary matrix
    rows = []
    for eq in equations:
        row = [int(b) for b in eq]
        rows.append(row)

    A = np.array(rows, dtype=int) % 2

    # Gaussian elimination mod 2
    pivot_cols = []
    pivot_row = 0
    for col in range(n):
        found = None
        for row in range(pivot_row, len(A)):
            if A[row, col] == 1:
                found = row
                break
        if found is None:
            continue
        A[[pivot_row, found]] = A[[found, pivot_row]]
        pivot_cols.append(col)
        for row in range(len(A)):
            if row != pivot_row and A[row, col] == 1:
                A[row] = (A[row] + A[pivot_row]) % 2
        pivot_row += 1

    # If there are free columns, choose s in the null space
    free_cols = [c for c in range(n) if c not in pivot_cols]
    if not free_cols:
        return '0' * n  # s = 0 implies a 1-to-1 function

    # Reconstruct s using the first free column
    s = np.zeros(n, dtype=int)
    s[free_cols[0]] = 1
    for i, col in enumerate(pivot_cols):
        if i < len(A) and A[i, free_cols[0]] == 1:
            s[col] = 1

    return ''.join(str(b) for b in s)


print('Simon functions defined.')

# Demo
secret = '110'
n = len(secret)
backend = AerSimulator()
qc_simon = simon_circuit(secret)
print(qc_simon.draw('text'))

In [ ]:
# Run several times to collect linear equations
secret = '1101'
n = len(secret)
qc_simon = simon_circuit(secret)

# We need at least n-1 linearly independent equations
backend = AerSimulator()
job = backend.run(qc_simon, shots=50)
counts = job.result().get_counts()

# Filter out the all-zeros result (non-informative)
equations = [eq[::-1] for eq in counts.keys() if eq != '0' * n]

print(f'Secret period: s = {secret}')
print(f'Collected equations (y·s = 0 mod 2):')
for eq in equations[:8]:
    dot = sum(int(a)*int(b) for a,b in zip(eq, secret)) % 2
    print(f'  y = {eq}, y·s = {dot}')

s_recovered = simon_postprocessing(equations, n)
print(f'\nRecovered period: s = {s_recovered}')
print(f'Correct: {s_recovered == secret}')

## 3C.2 Proposed Exercises

1. How many quantum measurements are sufficient on average to obtain $n-1$ linearly independent equations? Analyze the failure probability after $k$ measurements.

2. Implement Simon's oracle for the case $\mathbf{s} = \mathbf{0}$ (injective function) and verify that the algorithm returns the zero vector.

3. Can Simon's algorithm solve the discrete logarithm problem? Investigate the connections.